In [121]:
import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Allow running from either the project root or the test folder.
ROOT = Path.cwd()
if not (ROOT / "src/allocations.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# Pick up edits to allocations.py when this cell is rerun.
import src.allocations

importlib.reload(src.allocations)
from src.allocations import Allocations

In [ ]:
method = "test_bootstrap_nn"
TEST_SIZE = 500
SELECTION_RATE = 0.10
QUALIFICATION_COLUMN = "qualified_gagne_2"
NEED_COLUMN = "gagne_sum_t"
GROUP_COLUMN = "race"
RASHOMON_EPSILON = 0.01
ALLOCATION_DELTA = 0.1
NUM_MODELS = 20
ITERATIONS_PER_SPLIT = 5

fp_input = ROOT / "data/obermeyer/obermeyer_data_cleaned.csv"
fp_predictions = ROOT / f"data/predictions/{method}_{QUALIFICATION_COLUMN}.csv"
fp_output = ROOT / f"data/results/{method}_{QUALIFICATION_COLUMN}_dp_comparison.csv"

In [123]:
data = pd.read_csv(fp_input)
df = pd.read_csv(fp_predictions)
df = df.merge(
    data[["person_id", GROUP_COLUMN]],
    on="person_id",
    how="left",
    validate="many_to_one",
)

In [ ]:
a = Allocations(
    df=df,
    TEST_SIZE=TEST_SIZE,
    SELECTION_RATE=SELECTION_RATE,
    QUALIFICATION_COLUMN="y",
    NEED_COLUMN=NEED_COLUMN,
    GROUP_COLUMN=GROUP_COLUMN,
    RASHOMON_EPSILON=RASHOMON_EPSILON,
    ALLOCATION_EPSILON=ALLOCATION_DELTA,
    NUM_MODELS=NUM_MODELS,
    ITERATIONS_PER_SPLIT=ITERATIONS_PER_SPLIT,
)
allocations_opt, allocations_lda, people = a.get_allocations()
print(f"{len(people)} cohorts, {TEST_SIZE} people per cohort, k = {a.k}")

[Split 6]:   0%|          | 0/5 [00:00<?, ?it/s]

## Calculate Utility and Disparity

For each cohort, we look up each allocation's selected people by `person_id` and calculate:

- **Expected utility:** average probability under the model that produced the allocation.
- **Realized utility:** average observed qualification label `y` among selected people.
- **Absolute disparity:** absolute difference between the selected mean needs of groups 1 and 0.
- **Disparity ratio:** selected mean need of group 1 divided by that of group 0.

A missing selected group makes disparity undefined; a zero group-0 mean also makes the ratio undefined. These values are recorded as `NaN`. Our class currently falls back to top-k if DP finds no feasible solution, so a row labeled `DP` can contain that fallback allocation.

In [ ]:
all_results = []

for _, cohort_info in people.iterrows():
    seed = cohort_info["seed"]
    iteration = cohort_info["iteration"]

    split_df = df.loc[(df["seed"] == seed) & (df["person_id"] >= 0)]
    split_df = split_df.set_index("person_id")
    cohort = split_df.loc[cohort_info["people"]]

    for algorithm, allocations in [("OPT", allocations_opt), ("LDA", allocations_lda)]:
        cohort_allocations = allocations.loc[(allocations["seed"] == seed) & (allocations["iteration"] == iteration)]

        for _, allocation in cohort_allocations.iterrows():
            selected = cohort.loc[allocation["selected"]]

            mean_0 = selected.loc[selected[GROUP_COLUMN] == 0, NEED_COLUMN].mean()
            mean_1 = selected.loc[selected[GROUP_COLUMN] == 1, NEED_COLUMN].mean()
            absolute_disparity = abs(mean_1 - mean_0)
            disparity_ratio = np.nan
            if pd.notna(mean_0) and pd.notna(mean_1) and mean_0 != 0:
                disparity_ratio = mean_1 / mean_0

            for model in allocation["models"]:
                all_results.append({
                    "method": method,
                    "algorithm": algorithm,
                    "qualification_column": QUALIFICATION_COLUMN,
                    "seed": seed,
                    "iteration": iteration,
                    "model": model,
                    "k": a.k,
                    "epsilon": ALLOCATION_DELTA,
                    "tolerance_type": "add",
                    "expected_utility": selected[model].mean(),
                    "realized_utility": selected["y"].mean(),
                    "absolute_disparity": absolute_disparity,
                    "disparity_ratio": disparity_ratio,
                })

results = pd.DataFrame(all_results)
results

,method,algorithm,qualification_column,seed,iteration,model,k,epsilon,tolerance_type,expected_utility,realized_utility,absolute_disparity,disparity_ratio
0,test_bootstrap_nn,OPT,qualified_gagne_1,0,0,m_1,50,0.1,add,0.978008,0.98,1.375000,1.291005
1,test_bootstrap_nn,OPT,qualified_gagne_1,0,0,m_2,50,0.1,add,0.979383,0.98,2.229236,1.481694
2,test_bootstrap_nn,OPT,qualified_gagne_1,0,0,m_3,50,0.1,add,0.985311,0.98,1.232558,1.258537
3,test_bootstrap_nn,OPT,qualified_gagne_1,0,0,m_4,50,0.1,add,0.976809,0.98,1.969697,1.404984
4,test_bootstrap_nn,OPT,qualified_gagne_1,0,0,m_5,50,0.1,add,0.978705,0.98,1.417344,1.294980
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,test_bootstrap_nn,LDA,qualified_gagne_1,9,4,m_16,50,0.1,add,0.980456,0.96,0.000000,1.000000
1996,test_bootstrap_nn,LDA,qualified_gagne_1,9,4,m_17,50,0.1,add,0.971952,0.94,0.000000,1.000000
1997,test_bootstrap_nn,LDA,qualified_gagne_1,9,4,m_18,50,0.1,add,0.977641,0.96,0.000000,1.000000
1998,test_bootstrap_nn,LDA,qualified_gagne_1,9,4,m_19,50,0.1,add,0.978572,0.94,0.000000,1.000000


In [ ]:
results.groupby(["seed", "iteration", "method", "qualification_column", "algorithm"]).mean(True)

k  \
seed iteration method            qualification_column algorithm         
0    0         test_bootstrap_nn qualified_gagne_1    LDA        50.0   
                                                      OPT        50.0   
     1         test_bootstrap_nn qualified_gagne_1    LDA        50.0   
                                                      OPT        50.0   
     2         test_bootstrap_nn qualified_gagne_1    LDA        50.0   
...                                                               ...   
9    2         test_bootstrap_nn qualified_gagne_1    OPT        50.0   
     3         test_bootstrap_nn qualified_gagne_1    LDA        50.0   
                                                      OPT        50.0   
     4         test_bootstrap_nn qualified_gagne_1    LDA        50.0   
                                                      OPT        50.0   

                                                                 epsilon  \
seed iteration method            qualification_column algorithm            
0    0         test_bootstrap_nn qualified_gagne_1    LDA            0.1   
                                                      OPT            0.1   
     1         test_bootstrap_nn qualified_gagne_1    LDA            0.1   
                                                      OPT            0.1   
     2         test_bootstrap_nn qualified_gagne_1    LDA            0.1   
...                                                                  ...   
9    2         test_bootstrap_nn qualified_gagne_1    OPT            0.1   
     3         test_bootstrap_nn qualified_gagne_1    LDA            0.1   
                                                      OPT            0.1   
     4         test_bootstrap_nn qualified_gagne_1    LDA            0.1   
                                                      OPT            0.1   

                                                                 expected_utility  \
seed iteration method            qualification_column algorithm                     
0    0         test_bootstrap_nn qualified_gagne_1    LDA                0.978763   
                                                      OPT                0.980031   
     1         test_bootstrap_nn qualified_gagne_1    LDA                0.980995   
                                                      OPT                0.981301   
     2         test_bootstrap_nn qualified_gagne_1    LDA                0.981295   
...                                                                           ...   
9    2         test_bootstrap_nn qualified_gagne_1    OPT                0.973436   
     3         test_bootstrap_nn qualified_gagne_1    LDA                0.983586   
                                                      OPT                0.984165   
     4         test_bootstrap_nn qualified_gagne_1    LDA                0.980723   
                                                      OPT                0.982709   

                                                                 realized_utility  \
seed iteration method            qualification_column algorithm                     
0    0         test_bootstrap_nn qualified_gagne_1    LDA                   0.979   
                                                      OPT                   0.979   
     1         test_bootstrap_nn qualified_gagne_1    LDA                   0.990   
                                                      OPT                   0.990   
     2         test_bootstrap_nn qualified_gagne_1    LDA                   0.972   
...                                                                           ...   
9    2         test_bootstrap_nn qualified_gagne_1    OPT                   0.990   
     3         test_bootstrap_nn qualified_gagne_1    LDA                   0.999   
                                                      OPT                   0.999   
     4         test_bootstrap_nn qualified_gagne_1    LDA                   0.955   
   

In [ ]:
fp_output.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(fp_output, index=False)
print(f"Saved {len(results)} rows to {fp_output}")

Saved 2000 rows to /Users/kk985/PhD/Research/AllocationMultiplicity/data/results/test_bootstrap_nn_dp_comparison.csv
